# 05 - PCA y clustering

Objetivo: reproducir la lectura de clusters usando los CSVs versionados y analizar los segmentos resultantes sin depender de una conexion activa a PostgreSQL.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

customer_analytics = pd.read_csv(PROJECT_ROOT / "notebooks" / "customer_analytics.csv")
customer_clusters = pd.read_csv(PROJECT_ROOT / "notebooks" / "customer_clusters.csv")
df = customer_analytics.merge(customer_clusters, on="customer_id", how="left")
df.shape

In [ ]:
cluster_summary = df.groupby(["cluster_id", "cluster_name"]).agg(
    clientes=("customer_id", "count"),
    ingresos=("ingresos_netos", "sum"),
    margen=("margen_neto_estimado", "sum"),
    recencia_media=("recencia_dias", "mean"),
    tasa_devolucion_media=("tasa_devolucion_importe", "mean"),
    churn_medio=("churn_risk_score", "mean"),
).reset_index().sort_values("cluster_id")
cluster_summary.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for cluster_id, data in df.groupby("cluster_id"):
    ax.scatter(data["pca_1"], data["pca_2"], s=12, alpha=0.55, label=f"Cluster {cluster_id}")
ax.set_title("Clusters de clientes sobre PCA")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
pd.crosstab(df["cluster_name"], df["segmento_rfm"], normalize="index").round(3)

In [ ]:
df.sort_values(["cluster_id", "margen_neto_estimado"], ascending=[True, False]).groupby("cluster_id").head(5)[[
    "cluster_id",
    "cluster_name",
    "customer_id",
    "full_name",
    "margen_neto_estimado",
    "churn_risk_level",
    "segmento_rfm",
]]